# Algorithme « Metropolis à marches aléatoires » ("random-walk Metropolis").

## Importation des bibliothèques

In [2]:
# -q : quiet --> réduction des messages
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import time
import functions.generation as generation
from scipy.stats import invgamma, dirichlet

np.random.seed(28760)

In [4]:
# génération de données

x_star, mu, sigma2 = generation.generate_data(
    generation.n, generation.T, 
    generation.alpha, generation.zeta)

## Calcul de la vraisemblance

Le modèle de l’article est un modèle hiérarchique MA(2). Soit 

$$X = (X_1, X_2, \dots, X_T)$$


Chaque composante du vecteur $X$ est gaussienne, car elle est une combinaison linéaire de variables gaussiennes. Cependant, les composantes de ce vecteur ne sont pas indépendantes par construction.

$$
\mathrm{Var}(X_t)
= \mathrm{Cov}\big(y_t + \mu_1 y_{t-1} + \mu_2 y_{t-2},\; y_t + \mu_1 y_{t-1} + \mu_2 y_{t-2}\big)
$$

avec $y_t \sim \mathcal{N}(0, \sigma^2)$.

Par indépendance du bruit blanc (les innovations) :

$$
\mathrm{Var}(X_t) = \sigma^2 (1 + \mu_1^2 + \mu_2^2)
$$

On obtient les covariances suivantes :

$$
\mathrm{Cov}(X_t, X_{t-1}) = \mathrm{Cov}(X_t, X_{t+1})
= \sigma^2 (\mu_1 + \mu_1 \mu_2)
$$

$$
\mathrm{Cov}(X_t, X_{t-2}) = \mathrm{Cov}(X_t, X_{t+2})
= \sigma^2 \mu_2
$$

Ainsi, la matrice de variance-covariance de $X$, conditionnellement aux paramètres, est de la forme :

$$
\Sigma(\mu, \sigma^2) = \sigma^2
\begin{pmatrix}
\gamma_0 & \gamma_1 & \gamma_2 & 0 & \cdots & 0 \\
\gamma_1 & \gamma_0 & \gamma_1 & \gamma_2 & \ddots & \vdots \\
\gamma_2 & \gamma_1 & \gamma_0 & \gamma_1 & \ddots & 0 \\
0 & \gamma_2 & \gamma_1 & \gamma_0 & \ddots & \gamma_2 \\
\vdots & \ddots & \ddots & \ddots & \ddots & \gamma_1 \\
0 & \cdots & 0 & \gamma_2 & \gamma_1 & \gamma_0
\end{pmatrix}
$$